In [1]:
!pip install xgboost

In [2]:
from scipy.io import loadmat
import numpy as np
from sklearn.preprocessing import StandardScaler
import pandas as pd
import gc

from sklearn.linear_model import Ridge
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score


In [3]:
# Load all 3 subjects
sub1 = loadmat("sub1_comp.mat")
sub2 = loadmat("sub2_comp.mat")
sub3 = loadmat("sub3_comp.mat")

print("Sub1 train:", sub1['train_data'].shape, "-> 62 channels")
print("Sub2 train:", sub2['train_data'].shape, "-> 48 channels")
print("Sub3 train:", sub3['train_data'].shape, "-> 64 channels")

Sub1 train: (400000, 62) -> 62 channels
Sub2 train: (400000, 48) -> 48 channels
Sub3 train: (400000, 64) -> 64 channels


## Smart Channel Removal

We identify **actually unused/bad channels** using two criteria:

1. **Variance filter** — Remove dead/flat electrodes (near-zero variance) and noisy electrodes (extremely high variance)
2. **Correlation filter** — Remove channels that have zero or near-zero correlation with any of the 5 finger flexion signals

We apply both filters to each subject independently, then unify by keeping only the first N channels that survived in ALL subjects.

In [4]:
def find_good_channels(train_data, train_dg, subject_name,
                       var_low_percentile=1, var_high_percentile=99,
                       corr_threshold=0.01):
    """
    Identify good channels using variance and correlation filters.
    """
    n_channels = train_data.shape[1]

    # --- FILTER 1: Variance ---
    variances = np.var(train_data, axis=0)
    var_low = np.percentile(variances, var_low_percentile)
    var_high = np.percentile(variances, var_high_percentile)

    var_good = set()
    var_bad_low = []
    var_bad_high = []
    for ch in range(n_channels):
        if variances[ch] < var_low:
            var_bad_low.append(ch)
        elif variances[ch] > var_high:
            var_bad_high.append(ch)
        else:
            var_good.add(ch)

    print(f"\n{subject_name} - Variance Filter:")
    print(f"  Dead/flat channels (low var):  {var_bad_low}")
    print(f"  Noisy channels (high var):     {var_bad_high}")
    print(f"  Channels passing variance:     {len(var_good)}/{n_channels}")

    # --- FILTER 2: Correlation with finger flexion ---
    corr_good = set()
    corr_bad = []

    for ch in range(n_channels):
        ch_corrs = []
        for f in range(train_dg.shape[1]):
            r = np.abs(np.corrcoef(train_data[:, ch], train_dg[:, f])[0, 1])
            ch_corrs.append(r)
        max_corr = max(ch_corrs)

        if max_corr >= corr_threshold:
            corr_good.add(ch)
        else:
            corr_bad.append(ch)

    print(f"\n{subject_name} - Correlation Filter:")
    print(f"  Channels with no finger correlation: {corr_bad}")
    print(f"  Channels passing correlation:        {len(corr_good)}/{n_channels}")

    # --- COMBINE: Must pass BOTH filters ---
    good_channels = sorted(var_good & corr_good)
    removed = sorted(set(range(n_channels)) - set(good_channels))

    print(f"\n{subject_name} - Combined Result:")
    print(f"  Good channels: {len(good_channels)}/{n_channels}")
    print(f"  Removed channels: {removed}")

    return good_channels

In [5]:
# Apply filters to each subject
good_ch_1 = find_good_channels(sub1['train_data'], sub1['train_dg'], "Subject 1")
good_ch_2 = find_good_channels(sub2['train_data'], sub2['train_dg'], "Subject 2")
good_ch_3 = find_good_channels(sub3['train_data'], sub3['train_dg'], "Subject 3")


Subject 1 - Variance Filter:
  Dead/flat channels (low var):  [23]
  Noisy channels (high var):     [54]
  Channels passing variance:     60/62

Subject 1 - Correlation Filter:
  Channels with no finger correlation: [12]
  Channels passing correlation:        61/62

Subject 1 - Combined Result:
  Good channels: 59/62
  Removed channels: [12, 23, 54]

Subject 2 - Variance Filter:
  Dead/flat channels (low var):  [47]
  Noisy channels (high var):     [20]
  Channels passing variance:     46/48

Subject 2 - Correlation Filter:
  Channels with no finger correlation: [16]
  Channels passing correlation:        47/48

Subject 2 - Combined Result:
  Good channels: 45/48
  Removed channels: [16, 20, 47]

Subject 3 - Variance Filter:
  Dead/flat channels (low var):  [42]
  Noisy channels (high var):     [57]
  Channels passing variance:     62/64

Subject 3 - Correlation Filter:
  Channels with no finger correlation: []
  Channels passing correlation:        64/64

Subject 3 - Combined Result:

In [6]:
# Unify: keep the minimum number of good channels across all subjects
N_UNIFIED = min(len(good_ch_1), len(good_ch_2), len(good_ch_3))

print(f"Good channels per subject:")
print(f"  Subject 1: {len(good_ch_1)} out of 62")
print(f"  Subject 2: {len(good_ch_2)} out of 48")
print(f"  Subject 3: {len(good_ch_3)} out of 64")
print(f"\nUnified channel count: {N_UNIFIED}")

final_ch_1 = good_ch_1[:N_UNIFIED]
final_ch_2 = good_ch_2[:N_UNIFIED]
final_ch_3 = good_ch_3[:N_UNIFIED]

print(f"\nFinal channel indices used:")
print(f"  Subject 1: {final_ch_1}")
print(f"  Subject 2: {final_ch_2}")
print(f"  Subject 3: {final_ch_3}")

Good channels per subject:
  Subject 1: 59 out of 62
  Subject 2: 45 out of 48
  Subject 3: 62 out of 64

Unified channel count: 45

Final channel indices used:
  Subject 1: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46]
  Subject 2: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46]
  Subject 3: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 43, 44, 45]


In [7]:
# Apply channel selection
sub1_train = sub1['train_data'][:, final_ch_1]
sub1_test  = sub1['test_data'][:, final_ch_1]
sub1_dg    = sub1['train_dg']

sub2_train = sub2['train_data'][:, final_ch_2]
sub2_test  = sub2['test_data'][:, final_ch_2]
sub2_dg    = sub2['train_dg']

sub3_train = sub3['train_data'][:, final_ch_3]
sub3_test  = sub3['test_data'][:, final_ch_3]
sub3_dg    = sub3['train_dg']

print(f"After smart channel removal ({N_UNIFIED} channels each):")
print(f"  Sub1 train: {sub1_train.shape}")
print(f"  Sub2 train: {sub2_train.shape}")
print(f"  Sub3 train: {sub3_train.shape}")

After smart channel removal (45 channels each):
  Sub1 train: (400000, 45)
  Sub2 train: (400000, 45)
  Sub3 train: (400000, 45)


## Downsample Before Merging

The original data is sampled at **1000 Hz**, which is much higher than needed for finger flexion prediction (the finger data itself is only 25 Hz, upsampled to 1000 Hz).

We downsample by a factor of 4 (to 250 Hz), which:
- Cuts memory usage by 4x
- Still preserves signals up to 125 Hz (well above the useful motor cortex frequency range)
- Makes all subsequent steps 4x faster

In [8]:
# Downsample by factor of 4: 1000 Hz -> 250 Hz
DOWNSAMPLE_FACTOR = 4

sub1_train_ds = sub1_train[::DOWNSAMPLE_FACTOR]
sub1_dg_ds    = sub1_dg[::DOWNSAMPLE_FACTOR]

sub2_train_ds = sub2_train[::DOWNSAMPLE_FACTOR]
sub2_dg_ds    = sub2_dg[::DOWNSAMPLE_FACTOR]

sub3_train_ds = sub3_train[::DOWNSAMPLE_FACTOR]
sub3_dg_ds    = sub3_dg[::DOWNSAMPLE_FACTOR]

print(f"After downsampling (factor={DOWNSAMPLE_FACTOR}):")
print(f"  Sub1: {sub1_train.shape} -> {sub1_train_ds.shape}")
print(f"  Sub2: {sub2_train.shape} -> {sub2_train_ds.shape}")
print(f"  Sub3: {sub3_train.shape} -> {sub3_train_ds.shape}")

# Free original full-size arrays from memory
del sub1, sub2, sub3
del sub1_train, sub2_train, sub3_train
del sub1_dg, sub2_dg, sub3_dg
gc.collect()
print("\nFreed original data from memory.")

After downsampling (factor=4):
  Sub1: (400000, 45) -> (100000, 45)
  Sub2: (400000, 45) -> (100000, 45)
  Sub3: (400000, 45) -> (100000, 45)

Freed original data from memory.


In [9]:
# Merge all 3 subjects
X_train_all = np.concatenate([sub1_train_ds, sub2_train_ds, sub3_train_ds], axis=0)
y_train_all = np.concatenate([sub1_dg_ds, sub2_dg_ds, sub3_dg_ds], axis=0)

# Free individual subject arrays
del sub1_train_ds, sub2_train_ds, sub3_train_ds
del sub1_dg_ds, sub2_dg_ds, sub3_dg_ds
gc.collect()

print(f"Merged train data: {X_train_all.shape}")
print(f"Merged train_dg:   {y_train_all.shape}")
print(f"\nMemory: ~{X_train_all.nbytes / 1e9:.2f} GB (was ~{X_train_all.shape[0] * DOWNSAMPLE_FACTOR * X_train_all.shape[1] * 8 / 1e9:.2f} GB before downsampling)")

Merged train data: (300000, 45)
Merged train_dg:   (300000, 5)

Memory: ~0.05 GB (was ~0.43 GB before downsampling)


## Band Power Feature Extraction

We compute **power spectral density** in 5 physiologically meaningful frequency bands for each channel using Welch's method.

| Band | Range | Relevance |
|------|-------|-----------|
| Delta | 0.5–4 Hz | slow cortical potentials |
| Theta | 4–8 Hz | motor planning |
| Alpha | 8–13 Hz | motor inhibition |
| Beta | 13–30 Hz | motor execution |
| High-Gamma | 70–120 Hz | **primary finger movement encoder** |

Features per sample: `5 bands × N_UNIFIED channels` → small, fast, and meaningful.

A **250-sample (1s) window** with 50% overlap is used — enough frequency resolution to separate all bands cleanly at 250 Hz.

In [10]:
from scipy.signal import welch

def band_power(psd, freqs, fmin, fmax):
    """Integrate PSD between fmin and fmax using the trapezoidal rule."""
    idx = np.logical_and(freqs >= fmin, freqs <= fmax)
    return np.trapz(psd[idx], freqs[idx])

BANDS = {
    'delta':      (0.5,  4),
    'theta':      (4,    8),
    'alpha':      (8,   13),
    'beta':       (13,  30),
    'high_gamma': (76, 100),
}

FS          = 250        # Hz after downsampling
WINDOW_SIZE = 250        # 1-second window
STEP        = 25         # new sample every 0.1s

def extract_band_power_batched(X, y, window_size=WINDOW_SIZE, step=STEP, fs=FS):
    """
    Slide a window over X, compute per-channel band power for each window.
    Output shape: (n_windows, n_channels * n_bands)
    """
    n_channels = X.shape[1]
    n_bands    = len(BANDS)
    n_windows  = (len(X) - window_size) // step

    X_out = np.zeros((n_windows, n_channels * n_bands), dtype=np.float32)
    y_out = np.zeros((n_windows, y.shape[1]),            dtype=np.float32)

    for w in range(n_windows):
        start = w * step
        seg   = X[start : start + window_size]          # (window_size, n_channels)

        feats = []
        for ch in range(n_channels):
            freqs, psd = welch(seg[:, ch], fs=fs, nperseg=window_size)
            for fmin, fmax in BANDS.values():
                feats.append(band_power(psd, freqs, fmin, fmax))

        X_out[w] = feats
        y_out[w] = y[start + window_size // 2]          # label at window center

        if w % 5000 == 0:
            print(f"  {w:,}/{n_windows:,} windows ({100*w/n_windows:.1f}%)", end='\r')

    print(f"  {n_windows:,}/{n_windows:,} windows (100.0%)  Done!")
    return X_out, y_out


print(f"Extracting band power features...")
print(f"  Window: {WINDOW_SIZE} samples = {WINDOW_SIZE/FS:.1f}s | Step: {STEP} samples = {STEP/FS:.1f}s")
print(f"  Input : {X_train_all.shape}")

X_feature, y_feature = extract_band_power_batched(X_train_all, y_train_all)

print(f"\nOutput:")
print(f"  X_feature : {X_feature.shape}")
print(f"  y_feature : {y_feature.shape}")
print(f"  Memory    : ~{X_feature.nbytes / 1e6:.1f} MB")


Extracting band power features...
  Window: 250 samples = 1.0s | Step: 25 samples = 0.1s
  Input : (300000, 45)


/tmp/ipykernel_6207/2465699339.py:6: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(psd[idx], freqs[idx])


  11,990/11,990 windows (100.0%)  Done!

Output:
  X_feature : (11990, 225)
  y_feature : (11990, 5)
  Memory    : ~10.8 MB


In [11]:

del X_train_all, y_train_all
gc.collect()
print("Freed raw data from memory.")

Freed raw data from memory.


In [12]:
# Scaler initialized
Scaler = StandardScaler()
print("Scaler ready. Will fit on X_train only after split to prevent data leakage.")


Scaler ready. Will fit on X_train only after split to prevent data leakage.


In [13]:

feature_names = [
    f"ch_{ch}_{band}"
    for ch in range(N_UNIFIED)
    for band in BANDS.keys()
]

print(f"Total features : {len(feature_names)}")
print(f"First 10       : {feature_names[:10]}")


Total features : 225
First 10       : ['ch_0_delta', 'ch_0_theta', 'ch_0_alpha', 'ch_0_beta', 'ch_0_high_gamma', 'ch_1_delta', 'ch_1_theta', 'ch_1_alpha', 'ch_1_beta', 'ch_1_high_gamma']


In [14]:


X_prev = np.zeros_like(X_feature)
X_prev[1:] = X_feature[:-1]

X_context = np.concatenate([X_feature, X_prev], axis=1)

print(f"Original features : {X_feature.shape[1]}")
print(f"After temporal ctx: {X_context.shape[1]}  (current + 1 lag window)")

lag_feature_names = [f"{name}_lag1" for name in feature_names]
feature_names_ctx = feature_names + lag_feature_names
print(f"Total feature names: {len(feature_names_ctx)}")


Original features : 225
After temporal ctx: 450  (current + 1 lag window)
Total feature names: 450


In [15]:
# Convert to DataFrame. it's using X_context (unscaled, band power + lag features)
data_df = pd.DataFrame(X_context, columns=feature_names_ctx)

print(data_df.head())


      ch_0_delta    ch_0_theta     ch_0_alpha     ch_0_beta  ch_0_high_gamma  \
0  144556.765625  9.469124e+05  226219.921875  1.679255e+06     43308.812500   
1  236853.140625  1.033059e+06  206947.109375  1.220163e+06     34334.468750   
2  327311.593750  1.188750e+06  216786.453125  9.439842e+05     27336.378906   
3  380343.125000  1.109484e+06  274710.250000  9.113947e+05     25126.396484   
4  561424.875000  7.597007e+05  325028.593750  9.928630e+05     25019.324219   

   ch_1_delta   ch_1_theta    ch_1_alpha     ch_1_beta  ch_1_high_gamma  ...  \
0  1924163.25  1307323.125  7.755974e+05  8.877499e+05     10108.915039  ...   
1  2390319.00  1941923.250  8.254576e+05  1.272542e+06      8528.620117  ...   
2  3058343.50  2750565.500  8.185122e+05  1.537017e+06      7648.950195  ...   
3  3134345.50  3551764.000  9.110990e+05  1.633908e+06      7496.291992  ...   
4  2154238.50  4435458.500  1.162836e+06  1.569489e+06      8462.188477  ...   

   ch_43_delta_lag1  ch_43_theta_lag1 

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    data_df, y_feature, test_size=0.2, random_state=42, shuffle=False
)

# Scale AFTER split — fit only on training data to prevent leakage
X_train = Scaler.fit_transform(X_train)
X_test  = Scaler.transform(X_test)   # transform only, never fit

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")


X_train: (9592, 450)
X_test:  (2398, 450)
y_train: (9592, 5)
y_test:  (2398, 5)


In [17]:
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor

model = MultiOutputRegressor(
    XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,
        random_state=42,
        tree_method='hist'   # memory-efficient
    ),
    n_jobs=1
)

model.fit(X_train, y_train)
y_pred_raw = model.predict(X_test)

print(f"y_pred shape: {y_pred_raw.shape}")


y_pred shape: (2398, 5)


In [18]:
# ── Ridge model(bc lower scoce, we have decided not to use so committed)
# alphas = [0.01, 0.1, 1, 10, 100]
# model = RidgeCV(alphas=alphas, scoring='neg_mean_squared_error', cv=5)
# model.fit(X_train, y_train)
# y_pred_raw = model.predict(X_test)


def smooth_predictions(y_pred, window=5):
    """Apply a causal moving average along the time axis per finger."""
    smoothed = np.copy(y_pred)
    for f in range(y_pred.shape[1]):
        kernel = np.ones(window) / window
        smoothed[:, f] = np.convolve(y_pred[:, f], kernel, mode='same')
    return smoothed

y_pred = smooth_predictions(y_pred_raw, window=5)

EVAL_FINGERS = [0, 1, 2, 4]
EVAL_NAMES   = ['Thumb', 'Index', 'Middle', 'Pinky']

def pearson_r(y_true, y_pred):
    """Pure NumPy Pearson correlation per finger."""
    correlations = []
    for i in range(y_true.shape[1]):
        t = y_true[:, i] - np.mean(y_true[:, i])
        p = y_pred[:, i] - np.mean(y_pred[:, i])
        r = np.sum(t * p) / (np.sqrt(np.sum(t**2)) * np.sqrt(np.sum(p**2)) + 1e-8)
        correlations.append(r)
    return correlations

correlations = pearson_r(y_test, y_pred)

print("\n[Results] Pearson Correlation per Finger:")
for i, name in zip(EVAL_FINGERS, EVAL_NAMES):
    print(f"  {name:8s}: r = {correlations[i]:.4f}")

mean_r = np.mean([correlations[i] for i in EVAL_FINGERS])
print(f"\nMean Pearson r (4 fingers): {mean_r:.4f}")



[Results] Pearson Correlation per Finger:
  Thumb   : r = 0.5452
  Index   : r = 0.4085
  Middle  : r = 0.3061
  Pinky   : r = 0.5920

Mean Pearson r (4 fingers): 0.4630
